In [1]:
import random
from torch.utils.data import DataLoader

def unendlicher_oxford_subroute_generator(dataset, batch_size, shuffle=True, num_workers=2):
    subroute_ids = list(dataset.subroutes.keys())
    
    while True:
        # 1. Mische die Reihenfolge der Subrouten für diese "Meta-Epoche"
        if shuffle:
            random.shuffle(subroute_ids)
            
        for subroute_id in subroute_ids:
            # 2. Schalte das Dataset auf die aktuelle Subroute um
            dataset.set_subroute(subroute_id)
            
            # 3. Erstelle einen frischen DataLoader für diese spezifische Subroute
            subroute_loader = DataLoader(
                dataset, 
                batch_size=batch_size, 
                shuffle=shuffle, 
                num_workers=num_workers,
                drop_last=True # Wichtig für stabile Batch-Größen im LMC-Loss
            )
            
            # 4. Jiele alle Batches dieser Subroute durch, bevor der Generator weitergeht
            for batch in subroute_loader:
                # Wir geben zusätzlich die subroute_id mit, falls du sie loggen willst
                yield batch, subroute_id

In [2]:
def unendlicher_dark_driving_generator(dataloader):
    while True:
        for batch in dataloader:
            yield batch

In [ ]:
# 1. Initialisierung der Generatoren
oxford_gen = unendlicher_oxford_subroute_generator(oxford_train_dataset, batch_size=16)
dark_driving_loader = DataLoader(dark_driving_train_dataset, batch_size=16, shuffle=True, num_workers=2)
dark_gen = unendlicher_dark_driving_generator(dark_driving_loader)

# Start-Modus festlegen
current_mode = "oxford" 
CHECK_INTERVAL = 500 # Alle 500 Iterationen wird evaluiert

for global_step in range(TOTAL_ITERATIONS):
    
    if current_mode == "oxford":
        # Hole den nächsten Batch aus der aktuellen Oxford-Subroute
        (night_img, day_img, label), subroute_id = next(oxford_gen)
        
        # Verschiebe auf GPU
        night_img, day_img, label = night_img.cuda(), day_img.cuda(), label.cuda()
        
        # Vorwärtspass & LMC Loss berechnen
        optimizer.zero_grad()
        features_night = model(night_img)
        # Hinweis: Beim LMC Loss auf Oxford nutzt du meistens nur die Night-Features für die Classification,
        # oder berechnest den Loss separat für Day und Night.
        loss = lmc_loss_criterion(features_night, label) 
        loss.backward()
        optimizer.step()
        
        # Logging
        if global_step % 5 == 0:
            print(f"[{global_step}] Oxford Modus (Subroute {subroute_id}) | LMC Loss: {loss.item():.4f}")

    elif current_mode == "dark_driving":
        # Hole den nächsten perfekt ausgerichteten Batch von DarkDriving
        night_img, day_img = next(dark_gen)
        night_img, day_img = night_img.cuda(), day_img.cuda()
        
        optimizer.zero_grad()
        # IKT Loss: Vergleiche den trainierbaren Night-Student mit dem gefrorenen Day-Expert
        features_night = model(night_img)
        with torch.no_grad():
            features_day_expert = frozen_day_expert(day_img)
            
        # Dein IKT-Loss: 1 - cos_sim
        cos_sim = torch.nn.functional.cosine_similarity(features_day_expert, features_night, dim=1)
        ikt_loss = torch.mean(1.0 - cos_sim)
        
        # Skalierung mit alpha
        loss = alpha * ikt_loss
        loss.backward()
        optimizer.step()
        
        if global_step % 5 == 0:
            print(f"[{global_step}] DarkDriving Modus | IKT Loss: {loss.item():.4f}")

    # --- DER SWITCH/VALIDIERUNGS-PUNKT ---
    if global_step > 0 and global_step % CHECK_INTERVAL == 0:
        # 1. Führe deine Cosine-Similarity Validierung auf der Val-Szene aus
        val_metrics = evaluate_cosine_similarity_profiles(model, val_dataset)
        
        # 2. Entscheide basierend auf deinen Kriterien, ob der Modus gewechselt werden soll
        if current_mode == "oxford":
            # Wenn z.B. die Day-Night Ähnlichkeit sinkt -> Wechsle zu DarkDriving, um die Nacht zu stützen
            if val_metrics['day_night_cos_sim'] < SCHWELLENWERT:
                current_mode = "dark_driving"
                print(f"\n>>> Switch zu DARK DRIVING bei Iteration {global_step} <<<\n")
        
        elif current_mode == "dark_driving":
            # Wenn das Alignment maximal ist, gehe zurück zu Oxford, um die Geometrie nachzuschärfen
            if val_metrics['day_night_cos_sim'] > REICHREICHT_ZIEL oder global_step_in_dark > MAX_DARK_STEPS:
                current_mode = "oxford"
                print(f"\n>>> Switch zu OXFORD bei Iteration {global_step} <<<\n")